# Пункт 3.1

## Загрузка библиотек

In [68]:
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC 
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
import json
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

warnings.filterwarnings('ignore')

## Загрузка данных

In [4]:
data = pd.read_csv("final_dataset.csv").drop(columns=["Unnamed: 0", "Unnamed: 0.1"])

data.head()

,track_id,DateTime,latitude,longitude,altitude,temp,region,water_dist,building_dist,green_dist,water_area,building_area,green_area,season,hour,water_and_fire_cluster
0,1,2022-03-23 03:47:32,28.526319,77.205764,233.448227,21.0,Region 1,483.748137,17.263341,219.293124,9.614001,1361.365668,9.068051,Весна,3,1
1,1,2022-03-23 03:47:33,28.526304,77.205761,233.366989,16.0,Region 1,484.978274,18.888186,220.963695,9.614001,1361.576094,9.068051,Весна,3,1
2,1,2022-03-23 03:47:34,28.526298,77.205760,233.334747,25.0,Region 1,485.481786,19.533009,221.631384,9.614001,1353.855885,9.068051,Весна,3,1
3,1,2022-03-23 03:47:35,28.526288,77.205757,233.280121,19.0,Region 1,486.250814,20.664223,222.748454,9.614001,1350.015010,9.068051,Весна,3,1
4,1,2022-03-23 03:47:36,28.526277,77.205755,233.228973,21.0,Region 1,487.167542,21.860118,223.973323,9.614001,1350.783267,9.068051,Весна,3,1


## Классификация кластеров (пожароопасность/затопления)

In [9]:
# Разделение данных на X и Y
X = data.drop(columns=["water_and_fire_cluster", "track_id", "DateTime", "region"])
y = data["water_and_fire_cluster"]

### Тест на сбалансированность классов

In [7]:
fig = px.histogram(
    data, x="water_and_fire_cluster",
    title='Частота встречаемости классов',
    color=y,
    text_auto=True
)

fig.show()

На графике видно, что у классов сильный дисбаланс, так что в дальнейшем используем SMOTE для уравновешивания классов.

### Преодобработка категориальных признаков

In [12]:
ohe_season = OneHotEncoder()
seasons = ohe_season.fit_transform(X[['season']])
seasons.toarray()

array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       ...,
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.]])

In [13]:
seasons_df=pd.DataFrame(seasons.toarray(), columns=ohe_season.get_feature_names_out())
X = pd.concat([X.drop(['season'],axis=1),seasons_df], axis=1)
X.head()

,latitude,longitude,altitude,temp,water_dist,building_dist,green_dist,water_area,building_area,green_area,hour,season_Весна,season_Зима,season_Лето,season_Осень
0,28.526319,77.205764,233.448227,21.0,483.748137,17.263341,219.293124,9.614001,1361.365668,9.068051,3,1.0,0.0,0.0,0.0
1,28.526304,77.205761,233.366989,16.0,484.978274,18.888186,220.963695,9.614001,1361.576094,9.068051,3,1.0,0.0,0.0,0.0
2,28.526298,77.205760,233.334747,25.0,485.481786,19.533009,221.631384,9.614001,1353.855885,9.068051,3,1.0,0.0,0.0,0.0
3,28.526288,77.205757,233.280121,19.0,486.250814,20.664223,222.748454,9.614001,1350.015010,9.068051,3,1.0,0.0,0.0,0.0
4,28.526277,77.205755,233.228973,21.0,487.167542,21.860118,223.973323,9.614001,1350.783267,9.068051,3,1.0,0.0,0.0,0.0


In [31]:
pickle.dump(ohe_season, open("Encoder_1.sav", 'wb'))

### Стандартизация данных

In [14]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [32]:
pickle.dump(scaler, open("Scaler_1.sav", 'wb'))

Некоторые алгоритмы классификации чувствительны к масштабу данных => стандартизируем их.

### Функция уравновешивания классов с помощью SMOTE

In [50]:
def get_smote_data(X, y):
    smote = SMOTE()
    X, y = smote.fit_resample(X, y)
    return X, y

SMOTE (Synthetic Minority Over-sampling Technique) — это алгоритм предварительной обработки данных, используемый для устранения дисбаланса классов в наборе данных. Он создаёт новые, синтетические примеры, которые помогают модели лучше понять и обобщить характеристики минорных классов.

Устранение дисбаланса классов путем генерации синтетических примеров минорного класса (не изменяет данные, делая их "реальными"). Дисбаланс классов может ухудшить качество модели, особенно для минорного класса.

### Обучение моделей

#### Общая функция обучения

In [56]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def train_cross_val_model(model):
    all_metrics = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y)):
        X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        X_train, y_train = get_smote_data(X_train, y_train)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        fold_metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_macro": precision_score(y_test, y_pred, average='macro', zero_division=0),
            "recall_macro": recall_score(y_test, y_pred, average='macro', zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average='macro'),
            "f1_weighted": f1_score(y_test, y_pred, average='weighted'),
        }

        all_metrics.append(fold_metrics)

    # Средние метрики
    avg_metrics = pd.DataFrame(all_metrics).mean()

    return model, avg_metrics

StratifiedKFold обеспечивает равномерное распределение классов в каждом фолде, что критично при дисбалансе. Кросс-валидация даёт надёжную оценку качества модели, снижает риск переобучения. SMOTE применяется только к обучающим данным, предотвращая утечку. Подход универсален и позволяет сравнивать модели по усреднённым метрикам.

#### Logistic Regression

In [57]:
log_reg = LogisticRegression(random_state=42)

log_reg, metrics = train_cross_val_model(log_reg)

metrics

accuracy           1.0
precision_macro    1.0
recall_macro       1.0
f1_macro           1.0
f1_weighted        1.0
dtype: float64

#### RandomForestClassfier

In [58]:
rfc = RandomForestClassifier(random_state=42)

rfc, metrics = train_cross_val_model(rfc)

metrics

accuracy           1.0
precision_macro    1.0
recall_macro       1.0
f1_macro           1.0
f1_weighted        1.0
dtype: float64

#### SVC

In [59]:
svc = SVC(random_state=42)

svc, metrics = train_cross_val_model(svc)

metrics

accuracy           1.0
precision_macro    1.0
recall_macro       1.0
f1_macro           1.0
f1_weighted        1.0
dtype: float64

Лучшей моделью оказалась LogisticRegression за счёт её быстрого обучения. Мы сохраняем эту модель. В качестве оценки предпочтение отдавалось weighted f1-score, macro f1-score и accuracy, и все 3 модели показали отличный результат.

In [60]:
pickle.dump(log_reg, open("Classification_Model_1.sav", 'wb'))

## Классификация кластеров (сложность эвакуации)

Данные уже предобработаны, нужно будет изучить распределение классов, предложить их синтезирование, дальше обучить на 3 моделях (тех же) и выбрать лучшую.

# Пункт 3.2

Непрерывное обучение будет реализовано с помощью Apache Airflow. 2 модели будут переобучаться параллельно для ускорения процесса, получать данные из базы данных (предполагается, что данные в базе данных будут обновляться). 

In [76]:
def get_encoded_data(data):
    ohe_season = pickle.load(open("Encoder_1.sav", "rb"))
    seasons = ohe_season.fit_transform(data[['season']])
    seasons_df = pd.DataFrame(seasons.toarray(), columns=ohe_season.get_feature_names_out())
    data = pd.concat([data.drop(['season'],axis=1), seasons_df], axis=1)
    return data

def get_scaled_data(data):
    scaler = pickle.load(open("Scaler_1.sav", "rb"))
    scaled_data = scaler.transform(data)

    return scaled_data

def get_smote_data(X, y):
    smote = SMOTE() 
    X, y = smote.fit_resample(X, y)
    return X, y


def train_cross_val_model(model):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    all_metrics = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y)):
        X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        X_train, y_train = get_smote_data(X_train, y_train)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        fold_metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_macro": precision_score(y_test, y_pred, average='macro', zero_division=0),
            "recall_macro": recall_score(y_test, y_pred, average='macro', zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average='macro'),
            "f1_weighted": f1_score(y_test, y_pred, average='weighted'),
        }

        all_metrics.append(fold_metrics)

    # Средние метрики
    avg_metrics = pd.DataFrame(all_metrics).mean().to_dict()

    return model, avg_metrics


def save_metrics_to_db(metrics, model_name):
    # Метки
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    metrics["timestamp"] = [now]
    metrics["model_name"] = [model_name]

    metrics = pd.DataFrame(metrics)

    conn = sqlite3.connect("database.db")
    
    try:
        metrics.to_sql("model_metrics", conn, if_exists="append")
    except:
        print("Ошибка при загрузке данных")


def retrain_water_fire_model():
    data = pd.read_csv("final_dataset.csv").drop(columns=["Unnamed: 0", "Unnamed: 0.1"])

    X = data.drop(columns=["water_and_fire_cluster", "track_id", "DateTime", "region"])
    y = data["water_and_fire_cluster"]
    X = get_encoded_data(X)
    X_scaled = get_scaled_data(X)

    log_reg = pickle.load(open("Classification_Model_1.sav", "rb"))

    log_reg, metrics = train_cross_val_model(log_reg)
    
    pickle.dump(log_reg, open("Classification_Model_1.sav", 'wb'))

    save_metrics_to_db(metrics, "Water_Fire_Model")

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

# Импортируем функции обучения двух моделей
from retrain_script import retrain_water_fire_model, retrain_evacuation_model

default_args = {
    'owner': 'airflow',
    'retries': 1,
    'retry_delay': timedelta(minutes=1),
}

with DAG(
    dag_id='retrain_models_dag',
    default_args=default_args,
    description='Параллельное обновление двух моделей: пожар/вода и эвакуация',
    start_date=datetime(2025, 4, 17),
    schedule_interval='*/15 * * * *',
    catchup=False,
) as dag:

    retrain_fire_water = PythonOperator(
        task_id='retrain_water_fire_model',
        python_callable=retrain_water_fire_model,
    )

    retrain_evacuation = PythonOperator(
        task_id='retrain_evacuation_model',
        python_callable=retrain_evacuation_model,
    )

    # Параллельное выполнение — не нужно связывать задачи
    [retrain_fire_water, retrain_evacuation]
